# RLS Governance POC — Step-by-Step Test Notebook

This notebook exercises the application built from `specification.md`, one step at a time.
Run cells **in order, top to bottom** — later steps depend on state created by earlier ones.

Each step prints a clear `PASS` / `FAIL` line (or an `assert`, which raises immediately if
something is wrong) so you can tell exactly where things stand before moving to the next cell.

**Sections:**

1. Setup — imports and path
2. Load configuration (`project.json`, environment JSON files, `check.txt` credentials)
3. Unit-level checks of pure logic (no network) — role overlay, verification, change
   detection, LRO handling
4. Run the full automated `pytest` suite
5. Live Microsoft Entra ID authentication check (needs real or working credentials)
6. Live Fabric API checks — Get Semantic Model, Get Definition (read-only)
7. **Discover which roles to deploy** — driven by the PR/git diff when one exists, falling
   back to every `.tmdl` file present in the project's `roles/` folder otherwise — then a
   dry-run preview of the overlay for each (no data sent to Fabric)
8. **Opt-in** live deployment of every discovered role in a single call (writes to Fabric) —
   off by default
9. Post-deployment verification of every deployed role

> Sections 5-9 require network access to `https://api.fabric.microsoft.com` and a tenant
> where the configured Service Principal has access to the target workspace/semantic model.
> Rotate the credentials in `check.txt` before relying on this for anything beyond your own
> disposable test workspace — see the security note from earlier in this project's history.


## 1. Setup

Add the repository root to `sys.path` so `src` imports resolve, regardless of which directory Jupyter was launched from.

In [ ]:
import os
import sys

# Walk up from the current working directory until we find specification.md,
# so this cell works whether Jupyter was launched from the repo root or elsewhere.
candidate = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(candidate, "specification.md")):
    parent = os.path.dirname(candidate)
    assert parent != candidate, "Could not locate repository root (specification.md not found)"
    candidate = parent

REPO_ROOT = candidate
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

os.chdir(REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print("PASS: repo root located and added to sys.path")


In [ ]:
# Import every module under test up front, so an import error surfaces immediately
# rather than partway through the notebook.
from src import authentication, change_detection, config, deploy, errors, fabric_client, lro, role_overlay, verify

print("PASS: all src modules imported successfully")


## 2. Load configuration

Loads `projects/sales/project.json`, all four environment files, and the sample
credentials/role from `check.txt`, using the real `src.config` loader code (not
reimplemented here) so this notebook is testing the actual application.


In [ ]:
project_config = config.load_project_config("projects/sales/project.json")
print(project_config)

assert project_config.project_key == "sales"
assert project_config.role_source_path == "projects/sales/roles/"
assert project_config.role_definition_path_prefix == "definition/roles/"
print("PASS: project.json loaded and matches expected values")


In [ ]:
dev_config = config.load_environment_config("projects/sales/environments/dev.json")
test_config = config.load_environment_config("projects/sales/environments/test.json")
uat_config = config.load_environment_config("projects/sales/environments/uat.json")
prod_config = config.load_environment_config("projects/sales/environments/prod.json")

for env_config in (dev_config, test_config, uat_config, prod_config):
    print(env_config)
    assert env_config.workspace_id, "workspaceId must not be empty"
    assert env_config.semantic_model_id, "semanticModelId must not be empty"

print("PASS: all four environment config files loaded")


In [ ]:
# check.txt is not valid JSON/.env — it's a small block of "name = value" Python-literal
# assignments followed by a sample RLS role. Parse only the assignment lines with a regex
# (no exec of file content) and pull out the values we need for the live-API sections below.
import re

CHECK_TXT_PATH = os.path.join(REPO_ROOT, "check.txt")

with open(CHECK_TXT_PATH, "r", encoding="utf-8") as f:
    check_txt_content = f.read()

assignment_pattern = re.compile(r'^(\w+)\s*=\s*"([^"]*)"', re.MULTILINE)
check_txt_values = dict(assignment_pattern.findall(check_txt_content))

print("Parsed keys from check.txt:", sorted(check_txt_values.keys()))

for required_key in ("tenant_id", "client_id", "secret", "WORKSPACE_ID", "semanticModelId"):
    assert required_key in check_txt_values, f"check.txt missing expected key: {required_key}"

print("PASS: check.txt parsed; required keys present")
print("NOTE: rotate these credentials before relying on this workspace for anything beyond disposable testing.")


## 3. Unit-level checks of pure logic (no network required)

These exercise the same code paths as `tests/`, but inline and printed, so you can see the
before/after state directly. Nothing in this section talks to Fabric.


### 3.1 Role overlay — REPLACE (existing role path)

In [ ]:
import base64
import copy


def make_part(path, text):
    return {
        "path": path,
        "payload": base64.b64encode(text.encode("utf-8")).decode("ascii"),
        "payloadType": "InlineBase64",
    }


sample_definition = {
    "parts": [
        make_part("definition/database.tmdl", "database definition unchanged"),
        make_part("definition/model.tmdl", "model definition unchanged"),
        make_part("definition/roles/abc.tmdl", "OLD abc role content"),
    ]
}

definition_before = copy.deepcopy(sample_definition)

result = role_overlay.overlay_role(sample_definition, "abc.tmdl", "NEW abc role content")

print("Operation:", result.operation)
print("Target path:", result.target_path)

assert result.operation == role_overlay.REPLACE
assert result.target_path == "definition/roles/abc.tmdl"

replaced_part = next(p for p in sample_definition["parts"] if p["path"] == "definition/roles/abc.tmdl")
decoded = base64.b64decode(replaced_part["payload"]).decode("utf-8")
assert decoded == "NEW abc role content"

print("PASS: existing role replaced in place, payload matches new content")


### 3.2 Role overlay — APPEND (new role path) + preservation invariant

This also checks the spec's most important invariant: every definition part that is **not** the target RLS path must be byte-for-byte identical before and after.

In [ ]:
result = role_overlay.overlay_role(sample_definition, "NewFinanceRole.tmdl", "finance role body")

print("Operation:", result.operation)
print("Target path:", result.target_path)
print("Parts after append:", [p["path"] for p in sample_definition["parts"]])

assert result.operation == role_overlay.APPEND
assert result.target_path == "definition/roles/NewFinanceRole.tmdl"
assert len(sample_definition["parts"]) == len(definition_before["parts"]) + 1

# Preservation invariant: every part that existed before and isn't a target path is untouched.
target_paths = {"definition/roles/abc.tmdl", "definition/roles/NewFinanceRole.tmdl"}
for before_part in definition_before["parts"]:
    if before_part["path"] in target_paths:
        continue
    after_part = next(p for p in sample_definition["parts"] if p["path"] == before_part["path"])
    assert before_part["path"] == after_part["path"]
    assert before_part["payload"] == after_part["payload"]

print("PASS: new role appended; all unrelated parts preserved byte-for-byte")


### 3.3 Duplicate role path is rejected

In [ ]:
duplicate_definition = {
    "parts": [
        make_part("definition/roles/dup.tmdl", "one"),
        make_part("definition/roles/dup.tmdl", "two"),
    ]
}

try:
    role_overlay.overlay_role(duplicate_definition, "dup.tmdl", "three")
    raise AssertionError("Expected DuplicateDefinitionPartError to be raised")
except errors.DuplicateDefinitionPartError as exc:
    print("Correctly raised:", exc)
    print("PASS: duplicate definition part path is rejected")


### 3.4 Change detection — add/modify allowed, delete/rename rejected

In [ ]:
allowed_changes = [
    {"path": "projects/sales/roles/abc.tmdl", "status": "modified"},
    {"path": "projects/sales/roles/NewFinanceRole.tmdl", "status": "added"},
    {"path": "projects/sales/project.json", "status": "modified"},  # ignored: not a role file
]

changed_roles = change_detection.detect_role_changes(allowed_changes, project_config.role_source_path)
print("Detected role changes:", changed_roles)
assert set(changed_roles) == {"abc.tmdl", "NewFinanceRole.tmdl"}
print("PASS: added/modified role files detected, non-role files ignored")


In [ ]:
disallowed_changes = [
    {"path": "projects/sales/roles/abc.tmdl", "status": "removed"},
]

try:
    change_detection.detect_role_changes(disallowed_changes, project_config.role_source_path)
    raise AssertionError("Expected RoleDeleteNotSupportedError to be raised")
except errors.RoleDeleteNotSupportedError as exc:
    print("Correctly raised:", exc)
    print("PASS: role deletion is rejected (deleting from Git does not delete the Fabric role)")


### 3.5 Post-deployment verification — match, mismatch, and line-ending normalization

In [ ]:
deployed_definition = {
    "parts": [make_part("definition/roles/abc.tmdl", "line one\r\nline two\r\n")]
}

# CRLF in the "deployed" payload vs LF in the "Git" content should still match.
assert verify.verify_role_deployment(deployed_definition, "definition/roles/abc.tmdl", "line one\nline two\n")
print("PASS: matching content (with normalized line endings) verifies successfully")

try:
    verify.verify_role_deployment(deployed_definition, "definition/roles/abc.tmdl", "something else entirely")
    raise AssertionError("Expected PostDeploymentValidationFailedError to be raised")
except errors.PostDeploymentValidationFailedError as exc:
    print("Correctly raised:", exc)
    print("PASS: mismatched content is correctly rejected")

try:
    verify.verify_role_deployment(deployed_definition, "definition/roles/does-not-exist.tmdl", "anything")
    raise AssertionError("Expected PostDeploymentValidationFailedError to be raised")
except errors.PostDeploymentValidationFailedError as exc:
    print("Correctly raised:", exc)
    print("PASS: missing target path is correctly rejected")


### 3.6 Long-running-operation (LRO) handler — Running → Succeeded, Failed, and Timeout

Uses fake `get_operation_state` / `get_operation_result` callables (and injected `sleep`/`clock`) so this runs instantly with no real waiting or network calls.

In [ ]:
# Running -> Running -> Succeeded
states = iter([{"status": "Running"}, {"status": "Running"}, {"status": "Succeeded"}])
sleeps_recorded = []

lro_result = lro.wait_for_operation(
    operation_id="demo-op-1",
    get_operation_state=lambda _id: next(states),
    get_operation_result=lambda _id: {"definition": {"parts": []}},
    retry_after=2,
    sleep=sleeps_recorded.append,
    clock=iter([0, 0, 0, 0]).__next__,
)
print("Result after Succeeded:", lro_result)
print("First wait respected Retry-After of 2 seconds:", sleeps_recorded[0] == 2)
assert lro_result == {"definition": {"parts": []}}
assert sleeps_recorded[0] == 2
print("PASS: LRO poller follows Running -> Succeeded and returns the result")


In [ ]:
try:
    lro.wait_for_operation(
        operation_id="demo-op-2",
        get_operation_state=lambda _id: {"status": "Failed", "error": {"message": "simulated failure"}},
        get_operation_result=lambda _id: {},
        retry_after=0,
        sleep=lambda _s: None,
        clock=iter([0, 0]).__next__,
    )
    raise AssertionError("Expected LroFailedError to be raised")
except errors.LroFailedError as exc:
    print("Correctly raised:", exc)
    print("PASS: a Failed operation state raises LroFailedError")


In [ ]:
fake_clock_values = iter([0, 100, 500, 1000])

try:
    lro.wait_for_operation(
        operation_id="demo-op-3",
        get_operation_state=lambda _id: {"status": "Running"},
        get_operation_result=lambda _id: {},
        retry_after=0,
        timeout_seconds=50,
        sleep=lambda _s: None,
        clock=lambda: next(fake_clock_values),
    )
    raise AssertionError("Expected LroTimeoutError to be raised")
except errors.LroTimeoutError as exc:
    print("Correctly raised:", exc)
    print("PASS: an operation stuck Running past the timeout raises LroTimeoutError")


## 4. Run the full automated test suite

Runs `pytest tests/ -v` as a subprocess and streams the output inline, so you get an
independent confirmation covering everything in Section 3 plus the mocked `FabricClient` and
`deploy_roles` orchestration tests (which simulate a full Fabric round trip in-memory, using a
fake response shape that mirrors the real `getDefinition` payload — `{"definition": {"parts":
[...]}}`).


In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

assert result.returncode == 0, "pytest suite failed — see output above"
print("PASS: full pytest suite passed")


## 5. Live authentication check (network required)

Attempts to acquire a real Microsoft Entra ID access token for the Service Principal using
`src.authentication.ServicePrincipalTokenProvider` — the exact class used in production/CI.


In [ ]:
# Edit these to point at a real Service Principal + tenant to exercise live sections 5-9.
# Defaults to the values parsed from check.txt.
tenant_id = check_txt_values["tenant_id"]
client_id = check_txt_values["client_id"]
client_secret = check_txt_values["secret"]
workspace_id = check_txt_values["WORKSPACE_ID"]
semantic_model_id = check_txt_values["semanticModelId"]

token_provider = authentication.ServicePrincipalTokenProvider(tenant_id, client_id, client_secret)

LIVE_CREDENTIALS_AVAILABLE = False
try:
    access_token = token_provider.get_access_token()
    LIVE_CREDENTIALS_AVAILABLE = True
    print(f"PASS: acquired access token ({len(access_token)} chars, not printed)")
except errors.AuthenticationFailedError as exc:
    print(f"Authentication failed: {exc}")
    print("LIVE_CREDENTIALS_AVAILABLE = False -- sections 6-9 below will be skipped")

print()
print("LIVE_CREDENTIALS_AVAILABLE =", LIVE_CREDENTIALS_AVAILABLE)


## 6. Live Fabric API checks (read-only)

Only runs if Section 5 obtained a real token. Calls **Get Semantic Model** then
**Get Semantic Model Definition** against the configured DEV workspace — both read-only, no
data is modified. Note the response shape: the parts list is nested under a `"definition"`
key (`response["definition"]["parts"]`), not at the top level — this is the exact shape that
`src/deploy.py` previously mishandled.


In [ ]:
client = fabric_client.FabricClient(token_provider)

if not LIVE_CREDENTIALS_AVAILABLE:
    print("SKIPPED: no live credentials (see Section 5)")
else:
    model_info = client.get_semantic_model(workspace_id, semantic_model_id)
    print("Semantic model info:", model_info)
    print("PASS: semantic model exists and is accessible")


In [ ]:
if not LIVE_CREDENTIALS_AVAILABLE:
    print("SKIPPED: no live credentials (see Section 5)")
else:
    live_definition_response = client.get_semantic_model_definition(workspace_id, semantic_model_id)
    live_definition = live_definition_response["definition"]
    part_paths = [p["path"] for p in live_definition["parts"]]
    print(f"Definition retrieved with {len(part_paths)} part(s):")
    for p in part_paths:
        print(" -", p)
    print("PASS: definition retrieved (LRO handled transparently if Fabric returned 202)")


## 7. Discover which roles to deploy, then dry-run preview each one

The application should never hardcode a single role file name. Instead it deploys whatever
the **pull request actually changed**:

- If this checkout has a meaningful diff against a base ref (i.e. this is a feature branch
  with committed role changes ahead of `origin/main`, mirroring a real PR), only those
  changed `.tmdl` files under `projects/sales/roles/` are deployed — using the same
  `src.change_detection.detect_role_changes` function the GitHub Actions workflow uses.
- Otherwise (e.g. testing locally on `main` with nothing committed yet, or no `origin/main`
  to diff against), it falls back to **every** `.tmdl` file currently present in
  `projects/sales/roles/` — a full baseline sync rather than a partial one.

Either way, this cell computes `roles_to_deploy: dict[filename -> content]` used by every
subsequent section — nothing here is hardcoded to a specific role name.


In [ ]:
import glob
import subprocess


def discover_changed_role_files(base_ref: str, role_source_path: str):
    """Mirror src/main.py's git-diff-driven change detection. Returns None if the diff
    itself isn't usable (no such ref, not enough history, etc.) rather than raising, so the
    caller can fall back cleanly."""
    try:
        output = subprocess.check_output(
            ["git", "diff", "--name-status", f"{base_ref}...HEAD"],
            cwd=REPO_ROOT,
            text=True,
            stderr=subprocess.STDOUT,
        )
    except subprocess.CalledProcessError:
        return None

    status_map = {"A": "added", "M": "modified", "D": "removed", "R": "renamed"}
    changed_files = []
    for line in output.splitlines():
        if not line.strip():
            continue
        parts = line.split("\t")
        code_letter = parts[0][0]
        path = parts[-1]
        changed_files.append({"path": path, "status": status_map.get(code_letter, "modified")})

    return change_detection.detect_role_changes(changed_files, role_source_path)


def discover_all_role_files(role_source_path: str):
    pattern = os.path.join(REPO_ROOT, role_source_path, "*.tmdl")
    return sorted(os.path.basename(p) for p in glob.glob(pattern))


role_file_names = discover_changed_role_files("origin/main", project_config.role_source_path)

if role_file_names:
    print(f"Using PR/diff-driven change set (vs origin/main): {role_file_names}")
else:
    role_file_names = discover_all_role_files(project_config.role_source_path)
    print(f"No committed role changes found against origin/main; falling back to all role files present: {role_file_names}")

assert role_file_names, f"No .tmdl role files found under {project_config.role_source_path}"

roles_to_deploy = {}
for name in role_file_names:
    full_path = os.path.join(REPO_ROOT, project_config.role_source_path, name)
    with open(full_path, "r", encoding="utf-8") as f:
        roles_to_deploy[name] = f.read()

print()
print(f"roles_to_deploy ({len(roles_to_deploy)}):", list(roles_to_deploy.keys()))
print("PASS: role set to deploy determined dynamically, not hardcoded")


In [ ]:
if not LIVE_CREDENTIALS_AVAILABLE:
    print("SKIPPED: no live credentials (see Section 5)")
else:
    preview_definition = copy.deepcopy(live_definition)
    preview_results = role_overlay.overlay_roles(
        preview_definition, roles_to_deploy, project_config.role_definition_path_prefix
    )

    print("Dry-run overlay decisions (nothing sent to Fabric):")
    for r in preview_results:
        print(f"  {r.target_path}: would {r.operation}")

    print()
    print("PASS: dry-run overlay computed for every discovered role")


## 8. Live deployment (writes to Fabric) — opt-in only

This is the only step in the notebook that **modifies** the Fabric semantic model. It calls
`src.deploy.deploy_roles` once with the *entire* `roles_to_deploy` dict from Section 7 —
matching spec section 24 (multiple role changes are combined into a single Get-Definition /
overlay / Update-Definition cycle, never one call per role). This is the same orchestration
used by the GitHub Actions workflow: Get Semantic Model -> Get Definition (x2, concurrency
check) -> overlay every changed role -> Update Definition once -> re-read -> verify every
role.

**It will not run unless you explicitly set `RUN_LIVE_DEPLOYMENT = True` below.** Only do this
against a DEV/test workspace you're comfortable modifying.


In [ ]:
RUN_LIVE_DEPLOYMENT = False  # <-- change to True deliberately to actually deploy

if not RUN_LIVE_DEPLOYMENT:
    print("SKIPPED: RUN_LIVE_DEPLOYMENT is False. Set it to True in this cell to run a real deployment.")
elif not LIVE_CREDENTIALS_AVAILABLE:
    print("SKIPPED: no live credentials (see Section 5)")
else:
    deployment_report = deploy.deploy_roles(
        token_provider=token_provider,
        workspace_id=workspace_id,
        semantic_model_id=semantic_model_id,
        roles=roles_to_deploy,
        project_key=project_config.project_key,
        environment=dev_config.environment,
        role_path_prefix=project_config.role_definition_path_prefix,
        fabric_client=client,
    )

    print("Deployment report:")
    for r in deployment_report.role_results:
        print(f"  {r.target_path}: {r.operation}")
    print("Verified:", deployment_report.verified)
    print("Success:", deployment_report.success)

    assert deployment_report.success
    print(f"PASS: live deployment of {len(roles_to_deploy)} role(s) completed and verified")


## 9. Post-deployment manual verification

Independently re-reads the definition (a fresh Get Definition call, not reusing any
in-memory state from Section 8) and confirms **every** deployed role's content matches its
Git file byte-for-byte, as a final sanity check.


In [ ]:
if not (LIVE_CREDENTIALS_AVAILABLE and RUN_LIVE_DEPLOYMENT):
    print("SKIPPED: requires a completed live deployment from Section 8")
else:
    post_definition = client.get_semantic_model_definition(workspace_id, semantic_model_id)["definition"]

    expected_roles = {
        project_config.role_definition_path_prefix + name: content
        for name, content in roles_to_deploy.items()
    }
    verified_paths = verify.verify_role_deployments(post_definition, expected_roles)

    for path in verified_paths:
        print(f"PASS: independently re-verified '{path}' matches Git content after deployment")


## Summary

| Section | What it proves | Requires network? |
|---|---|---|
| 2 | Config files load and are well-formed | No |
| 3 | Core logic (overlay, verify, change detection, LRO) is correct | No |
| 4 | Full automated test suite passes | No |
| 5 | Service Principal can authenticate to Entra ID | Yes |
| 6 | Configured workspace/semantic model is reachable | Yes |
| 7 | Role set to deploy is discovered from the PR/diff (or all roles as fallback), and previewed safely | Yes |
| 8 | End-to-end deployment of every discovered role succeeds in one call (opt-in, writes data) | Yes |
| 9 | Every deployed role independently verified against Git | Yes |

If Sections 2-4 all pass, the application logic itself is correct and network-independent.
Sections 5+ additionally validate real Fabric/Entra connectivity once you supply working
credentials.
